# 使用@tool装饰器定义工具
## 1、自定义工具描述：description
举例1：
函数使用@tool装饰器修饰以后，就是一个可以被模型识别的工具了
如下的程序报错了，因为：
在没有提供description参数的情况下，要求函数必须提供docstring

In [ ]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
@tool
def get_weather(city:str):
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

修改为：

In [ ]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
@tool
def get_weather(city:str):
    """
    获取城市天气
    :param city:
    :return:
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

举例2：使用description参数

In [4]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
@tool(description="获取具体城市的天气情况")
def get_weather(city:str):

    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气情况',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例3：
在同时声明docstring和description的情况下，description的优先级更高

In [5]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
@tool(description="获取具体城市的天气情况")
def get_weather(city:str):
    """
    获取城市天气
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气情况',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例4：
不使用 @tool 装饰器时，docstring不合法会被视为普通文本，作为 description ，但如果使用了 @tool 时 docstring 不合法，将会抛出异常


In [7]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
@tool(parse_docstring=True)
def get_weather(city:str):
    """
    获取城市天气

    Args:
        city:城市
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市天气',
        'parameters': {
            'properties': {'city': {'description': '城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

# 2、更改工具名称：name_or_callable
开发中，习惯使用函数名作为工具名称，不推荐自定义工具名称
举例：

In [8]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
@tool(parse_docstring=True,name_or_callable="getWeather")
def get_weather(city:str):
    """
    获取城市天气

    Args:
        city:城市
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '获取城市天气',
        'parameters': {
            'properties': {'city': {'description': '城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

# 3、自定义args_schema
## 3.1使用Pydantic模型定义
举例1：

In [10]:
from pydantic import BaseModel
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
class WeatherInput(BaseModel):
    city:str
@tool(args_schema=WeatherInput)
def get_weather(city:str):
    """
    获取城市天气
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市天气',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2：

In [11]:
from pydantic import BaseModel,Field
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
class WeatherInput(BaseModel):
    city:str=Field(
        description="具体的城市",
        default="北京"
    )
@tool(args_schema=WeatherInput)
def get_weather(city:str):
    """
    获取城市天气
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市天气',
        'parameters': {
            'properties': {'city': {'default': '北京', 'description': '具体的城市', 'type': 'string'}},
            'type': 'object'
        }
    }
}